# Update Serving Endpoint Configuration

This notebook updates the AI Gateway, telemetry configuration, and tags for an existing serving endpoint.

Use this when:
- The endpoint already exists and you need to update these settings
- Deploying with `updateAIGatewayOnly=true` to skip model registration

In [0]:
dbutils.widgets.text("endpoint_name", "epic-on-fhir-requests", "Endpoint Name")
dbutils.widgets.text("catalog", "main", "Catalog")
dbutils.widgets.text("schema", "default", "Schema")
dbutils.widgets.text("component", "epic-on-fhir", "Tag: Component")
dbutils.widgets.text("environment", "dev", "Tag: Environment")
dbutils.widgets.text("project", "epic-on-fhir-bundle", "Tag: Project")
dbutils.widgets.text("owner", "matthew.giglia@databricks.com", "Tag: Owner")

endpoint_name = dbutils.widgets.get("endpoint_name")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
tag_component = dbutils.widgets.get("component")
tag_environment = dbutils.widgets.get("environment")
tag_project = dbutils.widgets.get("project")
tag_owner = dbutils.widgets.get("owner")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import AiGatewayConfig, AiGatewayGuardrails, AiGatewayInferenceTableConfig, AiGatewayRateLimit, AiGatewayRateLimitKey, AiGatewayRateLimitRenewal, AiGatewayUsageTrackingConfig, EndpointTag, TelemetryConfig

w = WorkspaceClient()

print(f"Updating endpoint: {endpoint_name}")
print(f"Catalog: {catalog}, Schema: {schema}")

In [0]:
# Configure AI Gateway with inference tables, usage tracking, and rate limits
ai_gateway_config = AiGatewayConfig(
    inference_table_config=AiGatewayInferenceTableConfig(
        catalog_name=catalog,
        schema_name=schema,
        table_name_prefix=f"{endpoint_name.replace('-', '_')}_payload",
        enabled=True
    ),
    usage_tracking_config=AiGatewayUsageTrackingConfig(
        enabled=True
    ),
    guardrails=AiGatewayGuardrails(
        input=None,
        output=None
    ),
    rate_limits=[
        AiGatewayRateLimit(
            key=AiGatewayRateLimitKey.USER,
            renewal_period=AiGatewayRateLimitRenewal.MINUTE,
            calls=100
        )
    ]
)

print("Updating AI Gateway configuration...")
w.serving_endpoints.update_ai_gateway(
    name=endpoint_name,
    ai_gateway_config=ai_gateway_config
)
print("✓ AI Gateway configuration updated")

In [0]:
# Configure OpenTelemetry logging for requests, predictions, and metrics
telemetry_config = TelemetryConfig(
    otel_traces_enabled=True,
    otel_logs_enabled=True,
    otel_metrics_enabled=True,
    otel_traces_table_name=f"{catalog}.{schema}.{endpoint_name.replace('-', '_')}_traces",
    otel_logs_table_name=f"{catalog}.{schema}.{endpoint_name.replace('-', '_')}_logs",
    otel_metrics_table_name=f"{catalog}.{schema}.{endpoint_name.replace('-', '_')}_metrics"
)

print("Updating telemetry configuration...")
w.serving_endpoints.update_config(
    name=endpoint_name,
    telemetry_config=telemetry_config,
    served_entities=[],  # Empty - only updating telemetry
    traffic_config=None  # Not updating traffic
)
print("✓ Telemetry configuration updated")

In [0]:
# Update endpoint tags
tags = [
    EndpointTag(key="component", value=tag_component),
    EndpointTag(key="environment", value=tag_environment),
    EndpointTag(key="project", value=tag_project),
    EndpointTag(key="owner", value=tag_owner)
]

print("Updating endpoint tags...")
w.serving_endpoints.patch(
    name=endpoint_name,
    add_tags=tags
)
print("✓ Endpoint tags updated")

In [0]:
# Verify the updates
endpoint = w.serving_endpoints.get(endpoint_name)

print("\n=== Endpoint Configuration ===")
print(f"Name: {endpoint.name}")
print(f"State: {endpoint.state.config_update}")

if endpoint.ai_gateway:
    print("\n✓ AI Gateway configured")
    if endpoint.ai_gateway.inference_table_config:
        print(f"  - Inference tables: {endpoint.ai_gateway.inference_table_config.catalog_name}.{endpoint.ai_gateway.inference_table_config.schema_name}.{endpoint.ai_gateway.inference_table_config.table_name_prefix}_*")
    if endpoint.ai_gateway.usage_tracking_config:
        print(f"  - Usage tracking: {endpoint.ai_gateway.usage_tracking_config.enabled}")
    if endpoint.ai_gateway.rate_limits:
        for rl in endpoint.ai_gateway.rate_limits:
            print(f"  - Rate limit: {rl.calls} calls per {rl.renewal_period} per {rl.key}")

if endpoint.config and endpoint.config.telemetry_config:
    tc = endpoint.config.telemetry_config
    print("\n✓ Telemetry configured")
    print(f"  - Traces: {tc.otel_traces_table_name}")
    print(f"  - Logs: {tc.otel_logs_table_name}")
    print(f"  - Metrics: {tc.otel_metrics_table_name}")

if endpoint.tags:
    print("\n✓ Tags:")
    for tag in endpoint.tags:
        print(f"  - {tag.key}: {tag.value}")